In [17]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-pro")

In [25]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Load your environment variables (ensure your .env has GOOGLE_API_KEY)
load_dotenv()

# 2. Initialize the LLM with a current, active model name
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # <-- Updated string here! 
    temperature=0.7 
)

# 3. Test the connection
response = llm.invoke("What are the core differences between LangChain and LlamaIndex?")
print(response.content)

LangChain and LlamaIndex are both prominent frameworks in the LLM ecosystem, but they address different core problems and have distinct primary focuses. While there's some overlap in functionality (especially around RAG), understanding their main design principles is key.

Here's a breakdown of their core differences:

## Core Differences

| Feature             | LangChain                                                               | LlamaIndex                                                                |
| :------------------ | :---------------------------------------------------------------------- | :------------------------------------------------------------------------ |
| **Primary Focus**   | **Orchestration, multi-step reasoning, agents, connecting LLMs to *actions* and *external tools*.** | **Data ingestion, indexing, retrieval, and providing *context* to LLMs from private/external data (RAG).** |
| **Main Goal**       | Enable LLMs to perform complex tasks by breaking th

In [26]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

In [27]:
# pydantic class for structured output

class Chunk(BaseModel): 
    
    chunk_text: str
    summary: str
    
    
class Chunker(BaseModel):
    
    chunks: list[Chunk]

In [32]:
# define model
# define model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

llm_chunker = llm.with_structured_output(schema=Chunker)

In [33]:
# prompt for chunking

prompt = ChatPromptTemplate(messages=[
    ("system", 
     """You are an expert Text Chunker that splits the given text and outputs them as a 
     list of strings. You understand the natural topic boundaries of text and 
     also do not change the existing text. You just split the text where ever applicable.
     Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
    ("human",
     "Split the given text into chunks\nText: {text}")
], input_variables=["text"])

In [34]:
# chunking through llm

model_chain = prompt | llm_chunker

response = model_chain.invoke({"text": text})

In [35]:
response

Chunker(chunks=[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.", summary='This chunk discusses the transformative power of artificial intelligence, highlighting the sophistication of machine learning algorithms, the efficiency of deep learning models in data processing, and the brain-inspired structure of neural networks.'), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='This chunk focuses on the principles of Italian cuisine, particularly pasta preparation, emphasizing fresh ingr

In [36]:
response.chunks

[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.", summary='This chunk discusses the transformative power of artificial intelligence, highlighting the sophistication of machine learning algorithms, the efficiency of deep learning models in data processing, and the brain-inspired structure of neural networks.'),
 Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='This chunk focuses on the principles of Italian cuisine, particularly pasta preparation, emphasizing fresh ingredients, prope

In [37]:
len(response.chunks)

3

In [38]:
for chunk in response.chunks:
    print("Chunk Text:", chunk.chunk_text)
    print("Summary:", chunk.summary)
    print("-" * 50)

Chunk Text: Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structure.
Summary: This chunk discusses the transformative power of artificial intelligence, highlighting the sophistication of machine learning algorithms, the efficiency of deep learning models in data processing, and the brain-inspired structure of neural networks.
--------------------------------------------------
Chunk Text: The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.
Summary: This chunk focuses on the principles of Italian cuisine, particularly pasta preparation, emphas